# Phase 3, VCTK: Named Style Axes From Real Voices

## Why this exists

Phase 3 wants named style axes -- presentation (gender), age, emotion --
derived as directions in `style_ttl` space. The ten shipped presets carry
exactly one free label, gender, and it is not enough to derive anything from.
`py/phase3_presentation_stage1.py` estimated the presentation direction as the
class mean difference between the five `F*` and five `M*` presets and tested
it leave-one-out: **7 of 10 correct**
(`py/results/phase3/presentation_axis.json`). Under a coin flip that is
`P(X>=7 | n=10, p=0.5) = 0.172` -- not distinguishable from chance. The same
direction explains only **20.3%** of the between-preset variance. Ten
labelled points cannot locate a direction in a 6,144-dimensional space (24
active `style_ttl` rows x 256), so Phase 3 cannot derive **any** named axis --
gender, and a fortiori age or emotion, which carry no preset label at all --
without more labelled voices. VCTK has roughly 110 speakers with gender
labels, about ten times the sample this project has used so far, which is
the right order of magnitude to try.

Two results, both established elsewhere in this repo, make this tractable now
rather than merely hoped-for:

- **The preset-difference span is a validated basis, and a WavLM probe reads
  coefficients in it accurately.** Perturbing `style_ttl` inside the 9-dim
  span of the ten presets' pairwise differences and predicting the
  perturbation's coordinates from WavLM features gives mean per-component
  R^2 **0.9376**, against **0.7593** for a random control subspace of equal
  rank and amplitude, with the oracle ceiling at 1.0000
  (`py/results/phase2a/presetspan_probe_A.json` /
  `presetspan_probe_B.json`, via `py/phase2b_generate_presetspan.py` and
  `py/phase2b_subspace_probe.py`). A row-matched random control closed only
  about 20% of that gap. So the span is not an arbitrary coordinate system --
  the audio really does carry its coefficients -- and a probe trained on
  engine-rendered audio around that span should, in principle, read the same
  coefficients off *any* voice's audio, including a real one it has never
  seen synthesized.
- **The frozen ONNX graph is differentiable with respect to `style_ttl`.**
  `py/onnx2torch_patches.py` converts the graph to PyTorch via `onnx2torch`
  plus two generic version-compatibility patches, and its self-test confirms
  gradients reach all 12,800 elements of `style_ttl` through `text_encoder`
  with output agreement against `onnxruntime` under 1e-4. About 123 seconds
  per gradient step was measured on CPU for that graph, which is the entire
  reason gradient-based refinement is GPU work and not something to run on a
  laptop.

## Two paths, fast one first

**Path A (the one that unblocks Phase 3).** For each VCTK speaker, extract
WavLM layer 3-5 mean+std features over a handful of utterances and run the
already-existing ridge probe -- the exact one behind the 0.9376 result above
-- to predict that speaker's coordinates in the 9-dim preset-difference span.
This is a forward pass: minutes for the whole corpus, no gradients, no GPU
time beyond running WavLM itself. `py/phase3_presentation_stage1.py` and its
leave-one-out gate already exist and take an arbitrary set of labelled
`style_ttl`-shaped vectors, so applying them to ~110 VCTK-derived points is a
**re-run**, not new analysis code -- this notebook imports that module and
repoints its `PRESETS` / `M_PRESETS` / `F_PRESETS` globals rather than
reimplementing the gate.

**The honest caveat, stated here and re-checked in a cell below, not just
here:** the probe was fitted on audio the *engine itself* rendered from
styles inside a small ball (`eps=0.20`) around one preset (M1). Whether it
transfers to a real recorded voice, which may sit far outside that ball, is
exactly the untested assumption this whole path rests on. This notebook does
not take that on faith -- it reports the distribution of predicted
coefficients for VCTK speakers against the distribution the ten presets
themselves occupy in the same coordinates, and prints a warning in the
cell's own output, not just here, if real voices land far outside that
region.

**Path B (the quality upgrade, optional, clearly marked, skippable).**
Gradient refinement of `style_ttl` through the frozen graph via
`onnx2torch_patches.apply_patches()`, warm-started from Path A's
coefficients, minimising a duration-invariant mel-spectrogram statistic
against the speaker's own recording. This is genuinely experimental: only
`text_encoder`'s conversion has been numerically verified anywhere in this
repo (the self-test in `onnx2torch_patches.py`); converting
`vector_estimator` and `vocoder` the same way is untested until the
cross-check cell in this notebook runs. Path A alone answers the axis
question this notebook exists to unblock -- Path B is gated behind a single
`RUN_PATH_B = False` flag and every cell after it no-ops cleanly when that
flag is off.

## What this notebook does not do

It does not derive the age axis (no VCTK speaker carries an age label wide
enough to matter -- see `new-plan.md` Phase 4) and it does not derive emotion
(VCTK carries none). It is scoped to presentation, because that is the axis
VCTK's gender labels can actually support.

## References

- [new-plan.md](../new-plan.md) -- "Phase 3: Deriving the Axes" and "Phase 4:
  Datasets" (VCTK's role and limits), the design this notebook executes.
- `py/phase3_presentation_stage1.py` -- the leave-one-out gate, imported and
  re-run here against VCTK points instead of the ten presets, unchanged.
- `py/phase2b_generate_presetspan.py` -- builds the 9-dim preset-difference
  basis (`build_presetspan_basis`) and renders the calibration corpus this
  notebook's probe is trained on (`generate_condition`, condition
  `preset_span`).
- `py/phase2b_subspace_probe.py` / `py/phase2b_probe.py` -- the ridge-probe
  and oracle-ceiling machinery this notebook reuses rather than
  reimplementing (`build_xy`, `fit_ridge`, `per_component_r2`, `pooled_r2`,
  `mean_cosine`).
- `py/phase2b_wavlm_embed.py` / `py/phase2b_subspace_embed.py` -- WavLM-large
  feature extraction and RMS level-matching, reused verbatim for both the
  calibration corpus and the VCTK recordings.
- `py/onnx2torch_patches.py` -- the two generic onnx2torch compatibility
  patches Path B depends on; see its module docstring for exactly what each
  one fixes and what has and has not been numerically verified.
- [docs/phase2a_scaling_colab.ipynb](phase2a_scaling_colab.ipynb) and
  [docs/style_extraction_colab.ipynb](style_extraction_colab.ipynb) -- the
  Colab conventions (Drive layout, resumable checkpointing, GPU-vs-CPU
  staging, asset fetching) this notebook follows.
- [CREDITS / VCTK](https://datashare.ed.ac.uk/handle/10283/3443) -- CSTR
  VCTK Corpus 0.92, University of Edinburgh. Released under the Open Data
  Commons Attribution License (ODC-By) 1.0; carry attribution into anything
  derived from it.

In [ ]:
# 1. GPU check. WavLM extraction (Path A) needs it; style generation (the
# calibration corpus) does not and stays on CPU -- py/helper.py raises
# NotImplementedError("GPU mode is not fully tested") for GPU inference, and
# this notebook does not route around that. Path B, if enabled, needs the GPU
# for gradient steps through the converted graph.
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'No CUDA device. Runtime > Change runtime type > Hardware accelerator: '
        'GPU (T4 or better), then Runtime > Run all. WavLM feature extraction '
        '(Path A) is the reason this notebook needs a GPU runtime at all; '
        'calibration-corpus generation is CPU-bound regardless.')

DEVICE = 'cuda'
print('GPU: %s' % torch.cuda.get_device_name(0))
print('VRAM: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1024 ** 3))
print('torch: %s, CUDA: %s' % (torch.__version__, torch.version.cuda))


In [ ]:
# 2. Drive: the calibration corpus, VCTK's cached subset, and every result
# JSON live here so a disconnect costs minutes of resync, not hours of
# regeneration or a re-download of VCTK.
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')

WORKSPACE = Path('/content/drive/MyDrive/supertonic-phase3-vctk')
CALIB_DIR = WORKSPACE / 'calibration'          # preset-span rendered corpus
VCTK_CACHE = WORKSPACE / 'vctk_cache'          # speaker-info.txt + N flacs/speaker
RESULTS = WORKSPACE / 'results'
PATHB_DIR = WORKSPACE / 'pathb'
for d in (WORKSPACE, CALIB_DIR, VCTK_CACHE, RESULTS, PATHB_DIR):
    d.mkdir(parents=True, exist_ok=True)

VCTK_COEFFS_PATH = RESULTS / 'vctk_coefficients.json'      # Path A checkpoint, per speaker
GATE_RESULT_PATH = RESULTS / 'presentation_axis_vctk.json'  # final gate output

print('Workspace: %s' % WORKSPACE)


In [ ]:
# 3. The fork's phase2/phase3 scripts, plus the ONNX graphs and the ten
# shipped presets. snapshot_download rather than git-lfs -- see
# docs/style_extraction_colab.ipynb's cell 4: git-lfs over 392 MB is flaky in
# Colab. Set REPO_URL / REPO_REF to your own fork+branch if you are not
# running from the branch that carries these scripts.
import subprocess
import sys

REPO = Path('/content/supertonic')
REPO_URL = 'https://github.com/supertone-inc/supertonic.git'   # set to your fork
REPO_REF = 'claude/roadmap-next-steps-tx1aa8'                   # branch carrying py/phase2*.py, py/phase3*.py

if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF,
                    REPO_URL, str(REPO)], check=True)
REPO_PY = str(REPO / 'py')
if REPO_PY not in sys.path:
    sys.path.insert(0, REPO_PY)

needed = ['helper.py', 'phase2b_generate.py', 'phase2b_generate_subspace.py',
          'phase2b_generate_presetspan.py', 'phase2b_subspace_embed.py',
          'phase2b_subspace_probe.py', 'phase2b_probe.py', 'phase2a_ceiling_audit.py',
          'phase2b_wavlm_embed.py', 'phase3_presentation_stage1.py', 'onnx2torch_patches.py']
missing = [n for n in needed if not (REPO / 'py' / n).exists()]
if missing:
    raise FileNotFoundError(
        '%s missing from %s -- REPO_URL/REPO_REF do not point at a checkout '
        'carrying the phase2/phase3 scripts. Point REPO_REF at the branch that '
        'has them and rerun this cell.' % (missing, REPO_PY))
print('Repo scripts present: %s' % REPO_PY)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'onnxruntime==1.23.1', 'numpy>=1.26.0', 'soundfile>=0.12.1',
                'librosa>=0.10.0', 'PyYAML>=6.0', 'huggingface_hub', 'scipy',
                'scikit-learn', 'torch', 'torchaudio'], check=True)

from huggingface_hub import snapshot_download

ASSETS = Path('/content/supertonic-assets')
snapshot_download(repo_id='Supertone/supertonic-3', local_dir=str(ASSETS),
                  allow_patterns=['onnx/*', 'voice_styles/*'])
ONNX_DIR = str(ASSETS / 'onnx')
VOICE_STYLE_DIR = str(ASSETS / 'voice_styles')
missing_assets = [n for n in ('duration_predictor.onnx', 'text_encoder.onnx',
                              'vector_estimator.onnx', 'vocoder.onnx', 'tts.json',
                              'unicode_indexer.json')
                  if not (Path(ONNX_DIR) / n).exists()]
if missing_assets:
    raise FileNotFoundError('Missing ONNX assets: %s' % missing_assets)
print('ONNX: %s' % ONNX_DIR)
print('Presets: %s' % sorted(p.stem for p in Path(VOICE_STYLE_DIR).glob('*.json')))


## Configuration

Every knob that changes how much this notebook does, or how it decides
things, lives here. Nothing below this cell needs manual edits for a
top-to-bottom run.

- **`N_PER_COND`** -- how many `preset_span`-condition calibration clips to
  render (default 320, matching `phase2b_generate_presetspan.py`'s own
  `N_PER_COND` exactly, so the freshly-measured probe R^2 below is directly
  comparable to the validated 0.9376 figure rather than a different-N number
  that merely looks similar).
- **`N_UTTS_PER_SPEAKER`** -- how many VCTK recordings to average per speaker
  before probing. Set to **5**: enough utterances that per-utterance
  prosodic and channel noise mostly averages out of the pooled WavLM
  features (the project's own guidance is to average "many utterances per
  speaker so channel cancels while identity persists" -- `new-plan.md`,
  Phase 4), while keeping the selective VCTK download to about
  `110 speakers x 5 = 550` short clips rather than the full corpus (roughly
  400 utterances per speaker, tens of GB). This is a judgment call, not a
  measured optimum -- nothing here established that 5 is better than, say,
  8; it is a small number chosen for a bounded first pass.
- **`RUN_PATH_B`** -- off by default. Path A alone answers the axis
  question; see the markdown before Path B's cells for what turning this on
  costs and what it has not yet verified.

In [ ]:
# 4. Configuration.
# N_PER_COND, EPS, TEST_FRAC and SPEED for the calibration corpus are NOT
# independently settable here -- generate_condition() (Step 2) reads them off
# phase2b_generate_presetspan's own module globals (320 / 0.20 / 0.25 / 1.05),
# by design, so the freshly-rendered corpus stays comparable to
# presetspan_probe_A.json. This constant exists so Step 2 can assert the two
# agree rather than silently using whichever one the module happens to have.
N_PER_COND = 320
LAYERS = [3, 4, 5]      # WavLM layers, 1-based -- matches presetspan_probe_A.json exactly

N_UTTS_PER_SPEAKER = 5  # VCTK utterances averaged per speaker -- see markdown for why 5

RUN_PATH_B = False              # gradient refinement -- optional, GPU-heavy, unverified
                                # end-to-end (see Path B's markdown). Leave False for a
                                # Path-A-only run, which is what answers the axis question.
PATH_B_MAX_SPEAKERS = 3         # if RUN_PATH_B: refine at most this many speakers, so
                                # turning it on never silently commits to all ~110
PATH_B_N_STEPS = 30             # gradient steps per speaker if RUN_PATH_B

print('N_PER_COND=%d  N_UTTS_PER_SPEAKER=%d  RUN_PATH_B=%s'
      % (N_PER_COND, N_UTTS_PER_SPEAKER, RUN_PATH_B))


## Step 1: the preset-difference span is the target coordinate system

Everything downstream -- the calibration corpus, the probe, and what a VCTK
speaker's "coefficients" even mean -- is expressed in the 9-dim basis
`py/phase2b_generate_presetspan.py` builds from the tangent-projected
differences `P_i - P_{M1}` between the ten shipped presets, restricted to the
24 active `style_ttl` rows (Phase 0's row-locality result). Building it here,
rather than only inside the calibration-generation cell, means the exact
same `B_a`, `P` and `rank` are available later for reconstructing an
approximate `style_ttl` from a VCTK speaker's *predicted* coefficients, and
for comparing those predictions against the presets' own coefficients in the
same coordinates.

In [ ]:
# 5. Build the preset-span basis. Reused, not reimplemented -- this is the
# exact function behind the 0.9376 / 0.7593 result cited above.
import numpy as np

from phase2b_generate import ACTIVE_ROWS, PRESETS
from phase2b_generate_presetspan import BASE_PRESET, build_presetspan_basis
from helper import Style, load_text_to_speech, load_voice_style, timer

all_presets = {p: load_voice_style([str(Path(VOICE_STYLE_DIR) / f'{p}.json')]) for p in PRESETS}
base_style = all_presets[BASE_PRESET]
base_ttl = base_style.ttl.astype(np.float32)   # (1, 50, 256)
dp_ref = base_style.dp.copy()

B_a, P, RANK, extras = build_presetspan_basis(base_ttl, all_presets)
print('Base preset: %s' % BASE_PRESET)
print('Preset-span basis rank: %d (expected 9 = 10 presets - 1 base)' % RANK)
if RANK != len(PRESETS) - 1:
    print('WARNING: rank is not 9 -- the nine preset differences are not fully independent '
          'in the tangent space. Everything below still works (it uses RANK throughout, not '
          'a hardcoded 9), but the "9-dim" framing in the markdown above would need revising.')

# Each preset's own coordinates in this basis, for the distribution check later.
# Mirrors the tangent-projection step build_presetspan_basis performs internally.
PRESET_COEFFS = {BASE_PRESET: np.zeros(RANK)}
for p in extras['other_presets']:
    ttl_p = all_presets[p].ttl.astype(np.float64)[0, ACTIVE_ROWS, :]
    diff = ttl_p - P
    radial = np.einsum('rc,rc->r', diff, P)
    diff_tan = diff - radial[:, None] * P
    PRESET_COEFFS[p] = B_a @ diff_tan.reshape(-1)

PRESET_GENDER = {p: ('F' if p.startswith('F') else 'M') for p in PRESETS}
print('Presets: %s' % {p: PRESET_GENDER[p] for p in PRESETS})


## Step 2: render the calibration corpus (`preset_span` condition only)

This is the training data for Path A's probe: engine-rendered clips whose
`style_ttl` perturbation coefficients in the `B_a` basis are known exactly
(`c_realized`), paired with the audio those styles produced. Only the
`preset_span` condition from `phase2b_generate_presetspan.py` is rendered --
not `random_control` or `row_matched_random`, which exist in that script to
answer a different question (whether on-manifold directions beat random
ones) and are not needed to train this probe.

Generation is CPU-only (`load_text_to_speech(..., use_gpu=False)`; GPU
inference is unsupported in `py/helper.py`) and, at roughly 1-2 seconds per
clip, 320 clips is a matter of minutes -- not the multi-hour regime the other
notebooks' per-clip checkpointing exists for. So this stage checkpoints at
the whole-corpus level: if a previous run already wrote `manifest.json` and
`subspace.npz` for this condition, it is not re-rendered.

In [ ]:
# 6. Render (or reuse) the preset-span calibration corpus.
import numpy as np

from phase2b_generate_presetspan import generate_condition

tts = load_text_to_speech(ONNX_DIR, use_gpu=False)   # also reused by Path B later

calib_cond_dir = CALIB_DIR / 'preset_span'
calib_manifest_path = calib_cond_dir / 'manifest.json'
calib_subspace_path = calib_cond_dir / 'subspace.npz'

if calib_manifest_path.exists() and calib_subspace_path.exists():
    print('Calibration corpus already present at %s -- skipping generation.' % calib_cond_dir)
else:
    print('Rendering %d preset_span clips to %s (a few minutes on CPU)...' % (N_PER_COND, calib_cond_dir))
    sample_rng = np.random.default_rng(20260918)   # independent of phase2b_generate_presetspan's
                                                    # own SAMPLE_SEED -- this is a fresh corpus,
                                                    # not a re-use of a prior run's draws
    # generate_condition renders exactly N_PER_COND samples of the given condition using the
    # module's own N_PER_COND(=320)/TEST_FRAC(=0.25)/EPS(=0.20)/SPEED(=1.05) constants -- all of
    # which this notebook intentionally leaves untouched so the result stays comparable to
    # presetspan_probe_A.json. See that script's module docstring for why SPEED=1.05 here,
    # not this fork's speed=1.0 default.
    import phase2b_generate_presetspan as pgp
    assert pgp.N_PER_COND == N_PER_COND, (pgp.N_PER_COND, N_PER_COND)
    records, c_drawn_all, c_realized_all, ttl_all, _ = generate_condition(
        'preset_span', B_a, P, RANK, base_ttl, dp_ref, tts, sample_rng,
        idx_global_start=0, out_dir=str(calib_cond_dir), smoke=False)

    calib_cond_dir.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(calib_subspace_path,
                        basis=B_a.astype(np.float32), base_ttl=base_ttl[0],
                        active_rows=np.array(ACTIVE_ROWS),
                        **{f'c_realized_K{RANK}': c_realized_all, f'c_drawn_K{RANK}': c_drawn_all,
                           f'ttl_K{RANK}': ttl_all})
    import json as _json
    calib_manifest_path.write_text(_json.dumps({
        'experiment': 'phase3_vctk_styles_colab calibration corpus',
        'condition': 'preset_span', 'base_preset': BASE_PRESET, 'rank': RANK,
        'n_total': len(records), 'records': records,
    }, indent=2))
    bad = [r for r in records if not r['finite'] or r['peak'] >= 1.0]
    print('Wrote %d clips -> %s (%d non-finite/clipped)' % (len(records), calib_cond_dir, len(bad)))


## Step 3: WavLM-embed the calibration corpus and fit the ridge probe (GPU)

Same representation as `presetspan_probe_A.json`: WavLM-large layers 3, 4, 5
(1-based), mean+std pooled over frames, 6,144 dims. Fitting is reused
verbatim from `phase2b_probe.fit_ridge` / `phase2b_subspace_probe.build_xy`.
The train/test split this corpus was generated with reproduces (loosely --
different RNG draws, same design) the validated 0.9376 mean per-component
R^2; that comparison is printed below as a sanity check on this fresh
render, not as a strict pass/fail; the exact number will differ because the
random draws differ.

The probe actually deployed on VCTK is then refit on **all** 320 calibration
samples (train and test combined) -- there is no more held-out evaluation to
protect once the sanity check above has run, so the extra data goes into the
one probe that has to generalize the furthest.

In [ ]:
# 7. WavLM-large on GPU, calibration corpus.
import time

import soundfile as sf
import torch as _torch

from phase2b_subspace_embed import TARGET_RMS, level_match
from phase2b_wavlm_embed import DIM, N_LAYERS, load_model, pooled_features

bundle, wavlm_model = load_model()
wavlm_model = wavlm_model.to(DEVICE)
NORMALIZE = bool(getattr(bundle, '_normalize_waveform', True))
print('WavLM-large on %s; normalize_waveform=%s' % (DEVICE, NORMALIZE))


def pooled_features_gpu(model, normalize, wav_array, device):
    # Same forward pass / pooling as phase2b_wavlm_embed.pooled_features, on an
    # in-memory waveform array placed on `device` first.
    x = _torch.from_numpy(wav_array)[None, :].to(device)
    if normalize:
        x = _torch.nn.functional.layer_norm(x, x.shape)
    with _torch.no_grad():
        feats, _ = model.extract_features(x)
    assert len(feats) == N_LAYERS, len(feats)
    mean = np.empty((N_LAYERS, DIM), dtype=np.float32)
    std = np.empty((N_LAYERS, DIM), dtype=np.float32)
    for li, f in enumerate(feats):
        f = f[0]
        mean[li] = f.mean(0).cpu().numpy()
        std[li] = f.std(0).cpu().numpy()
    return mean, std


def embed_clips(records, audio_dir, out_path, rank):
    if out_path.exists():
        cached = np.load(out_path)
        if int(cached['n'][0]) == len(records):
            print('%s already has all %d clips embedded -- skipping.' % (out_path, len(records)))
            return cached
    mean_buf = np.zeros((len(records), N_LAYERS, DIM), dtype=np.float32)
    std_buf = np.zeros((len(records), N_LAYERS, DIM), dtype=np.float32)
    rms_buf = np.zeros(len(records), dtype=np.float32)
    idx_buf = np.zeros(len(records), dtype=np.int64)
    k_buf = np.full(len(records), rank, dtype=np.int64)
    n_kept = 0
    t0 = time.time()
    for i, r in enumerate(records):
        wav, sr = sf.read(str(Path(audio_dir) / r['file']), dtype='float32')
        assert sr == 16000
        leveled, rms_pre = level_match(wav)
        if leveled is None:
            continue
        mean, std = pooled_features_gpu(wavlm_model, NORMALIZE, leveled, DEVICE)
        mean_buf[n_kept], std_buf[n_kept], rms_buf[n_kept] = mean, std, rms_pre
        idx_buf[n_kept] = int(r['idx'])
        n_kept += 1
        if n_kept % 100 == 0 or (i + 1) == len(records):
            print('  %d/%d embedded  %.1f clips/sec' % (n_kept, len(records), n_kept / max(time.time() - t0, 1e-9)),
                  flush=True)
    mean_buf, std_buf = mean_buf[:n_kept], std_buf[:n_kept]
    rms_buf, idx_buf, k_buf = rms_buf[:n_kept], idx_buf[:n_kept], k_buf[:n_kept]
    np.savez_compressed(out_path, mean=mean_buf, std=std_buf, idx=idx_buf, K=k_buf,
                        rms_pre=rms_buf, target_rms=np.array([TARGET_RMS], dtype=np.float32),
                        n=np.array([n_kept]))
    print('Wrote %s: %d/%d clips' % (out_path, n_kept, len(records)))
    return np.load(out_path)


import json as _json
calib_meta = _json.loads(calib_manifest_path.read_text())
calib_records = calib_meta['records']
calib_feats = embed_clips(calib_records, calib_cond_dir / 'audio16k', calib_cond_dir / 'wavlm_feats.npz', RANK)


In [ ]:
# 8. Fit the probe: sanity-check the train/test split against the validated
# result, then refit on all 320 samples for deployment against VCTK.
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler

from phase2a_ceiling_audit import numerical_rank, oracle_projection
from phase2b_probe import ALPHAS, fit_ridge
from phase2b_subspace_probe import build_xy, mean_cosine, per_component_r2, pooled_r2

calib_subspace = np.load(calib_subspace_path)
X, Y, rms_pre, itr, ite, n_matched, n_missing = build_xy(
    RANK, calib_records, calib_feats, calib_subspace, LAYERS)
print('Calibration corpus: %d matched (%d missing from feats), %d train / %d test'
      % (n_matched, n_missing, len(itr), len(ite)))

Ypred, alpha = fit_ridge(X[itr], Y[itr], X[ite])
comp_r2 = per_component_r2(Y[ite], Ypred)
rank_eps, _, _ = numerical_rank(Y[itr])
P_oracle, _ = oracle_projection(Y[itr], Y[ite], rank_eps)
oracle_r2 = pooled_r2(Y[itr], Y[ite], P_oracle)

REFERENCE_A = {'source': 'py/results/phase2a/presetspan_probe_A.json',
               'per_component_r2_mean': 0.9376216839157002, 'oracle_r2': 1.0}
print('\nThis run   : mean per-component R^2 = %.4f  (oracle ceiling %.4f)'
      % (float(np.nanmean(comp_r2)), oracle_r2))
print('Reference  : mean per-component R^2 = %.4f  (%s)'
      % (REFERENCE_A['per_component_r2_mean'], REFERENCE_A['source']))
if np.nanmean(comp_r2) < 0.75:
    print('WARNING: the probe fitted in this run underperforms the validated reference by a '
          'wide margin -- something about this render (corpus size, WavLM version, layer '
          'indexing) may differ from the reference run. Treat every VCTK prediction below with '
          'extra caution.')
else:
    print('Broadly reproduces the validated result (exact match is not expected -- different RNG draws).')

# Deployed probe: refit on ALL 320 calibration samples. No more held-out
# evaluation to protect past this point, so use every sample available.
DEPLOYED_SCALER = StandardScaler().fit(X)
DEPLOYED_MODEL = RidgeCV(alphas=ALPHAS)
DEPLOYED_MODEL.fit(DEPLOYED_SCALER.transform(X), Y)
print('\nDeployed probe refit on all %d calibration samples (alpha=%.4g).'
      % (len(X), DEPLOYED_MODEL.alpha_))


**Caveat, restated because it matters more here than anywhere else in this
notebook:** the probe above was fitted entirely on audio the engine
synthesized from styles inside a small ball (`eps=0.20`, roughly a few
degrees of per-row rotation) around one preset. Nothing has shown yet that
it reads coefficients correctly for a voice that sits far outside that ball
-- which every real, unseen VCTK speaker might. Step 6 below is the check;
its warning is not decorative.

## Step 4: fetch VCTK

**What actually works from Colab, and what this notebook uses:** the
official University of Edinburgh DataShare release, downloaded directly with
`wget` --
`https://datashare.ed.ac.uk/bitstream/handle/10283/3443/VCTK-Corpus-0.92.zip`
(about 11.75 GB). This is the same URL a large number of published TTS
recipes use to fetch VCTK, requires no authentication, and needs no third
-party mirror whose contents cannot be checked. It has **not been executed
from inside this exact session** -- this notebook was authored without a
live Colab runtime to test against, so treat the download step as the one
part of this pipeline most likely to need a small fix (a redirect, a
renamed asset) if DataShare has changed since. If it fails outright, the
fallback is manual: download the zip yourself, drop it at
`WORKSPACE / 'VCTK-Corpus-0.92.zip'` (on Drive), and rerun this cell -- it
checks for a local copy before attempting the network fetch.

The zip is large but this notebook does not need most of it: only
`speaker-info.txt` (for speaker ID, age, and gender) and `N_UTTS_PER_SPEAKER`
`_mic1.flac` files per speaker from `wav48_silence_trimmed/`. Everything
needed is extracted selectively with `zipfile` and copied into the Drive
cache (a few hundred MB); the multi-GB zip itself is deleted afterward
rather than kept. **Resumability note:** the zip download itself is not
checkpointed byte-by-byte across a full session restart (`wget -c` resumes
within one still-running session, but a fresh Colab VM has no local file to
resume from) -- what *is* checkpointed, and is the part actually likely to
span multiple sessions, is everything from Step 5 onward, per VCTK speaker.
If the Drive cache from a previous run is already complete, this cell
detects that and skips the download+extraction entirely.

VCTK is CSTR VCTK Corpus 0.92 (University of Edinburgh), Open Data Commons
Attribution License (ODC-By) 1.0 -- carry attribution into anything derived
from styles fitted here.

In [ ]:
# 9. Fetch VCTK: speaker-info.txt + N_UTTS_PER_SPEAKER mic1 utterances/speaker.
import shutil
import zipfile

VCTK_ZIP_URL = 'https://datashare.ed.ac.uk/bitstream/handle/10283/3443/VCTK-Corpus-0.92.zip'
VCTK_LOCAL_ZIP = Path('/content/VCTK-Corpus-0.92.zip')
VCTK_WORKSPACE_ZIP = WORKSPACE / 'VCTK-Corpus-0.92.zip'   # manual-fallback location
SPEAKER_INFO_CACHE = VCTK_CACHE / 'speaker-info.txt'


def parse_speaker_info(path):
    # {speaker_id: {'age': str, 'gender': 'M'|'F'}} -- skips the header and any
    # row whose gender is not exactly M or F (VCTK's file has none, but be safe).
    out = {}
    lines = Path(path).read_text(errors='replace').splitlines()
    for line in lines[1:]:
        parts = line.split()
        if len(parts) < 3:
            continue
        sid, age, gender = parts[0], parts[1], parts[2]
        if gender not in ('M', 'F'):
            continue
        out[sid] = {'age': age, 'gender': gender}
    return out


def vctk_cache_complete():
    if not SPEAKER_INFO_CACHE.exists():
        return False
    spk = parse_speaker_info(SPEAKER_INFO_CACHE)
    if not spk:
        return False
    for sid in spk:
        d = VCTK_CACHE / sid
        if not d.exists() or len(list(d.glob('*_mic1.flac'))) < N_UTTS_PER_SPEAKER:
            return False
    return True


if vctk_cache_complete():
    print('Drive cache at %s already has speaker-info.txt and >= %d mic1 utterances for every '
          'listed speaker -- skipping download and extraction.' % (VCTK_CACHE, N_UTTS_PER_SPEAKER))
else:
    if not VCTK_LOCAL_ZIP.exists():
        if VCTK_WORKSPACE_ZIP.exists():
            print('Using manually-provided zip at %s' % VCTK_WORKSPACE_ZIP)
            VCTK_LOCAL_ZIP = VCTK_WORKSPACE_ZIP
        else:
            print('Downloading %s (about 11.75 GB; wget -c resumes if this cell is rerun '
                  'within the same session)...' % VCTK_ZIP_URL)
            subprocess.run(['wget', '-c', '-q', '--show-progress', '-O', str(VCTK_LOCAL_ZIP),
                            VCTK_ZIP_URL], check=True)

    with zipfile.ZipFile(VCTK_LOCAL_ZIP) as zf:
        names = zf.namelist()
        try:
            info_name = next(n for n in names if n.endswith('speaker-info.txt'))
        except StopIteration:
            raise RuntimeError(
                'No speaker-info.txt found inside the zip -- its internal layout may have '
                'changed. First few entries: %s' % names[:10])
        prefix = info_name[: -len('speaker-info.txt')]   # '' or e.g. 'VCTK-Corpus-0.92/'
        print('Zip internal prefix: %r' % prefix)

        extract_tmp = Path('/content/vctk_extract')
        extract_tmp.mkdir(exist_ok=True)
        zf.extract(info_name, extract_tmp)
        shutil.copy2(extract_tmp / info_name, SPEAKER_INFO_CACHE)

        speakers_all = parse_speaker_info(SPEAKER_INFO_CACHE)
        print('speaker-info.txt lists %d speakers with an M/F gender label' % len(speakers_all))

        wav_root = f'{prefix}wav48_silence_trimmed/'
        for i, (sid, info) in enumerate(speakers_all.items()):
            spk_dir = VCTK_CACHE / sid
            spk_dir.mkdir(exist_ok=True)
            existing = len(list(spk_dir.glob('*_mic1.flac')))
            if existing >= N_UTTS_PER_SPEAKER:
                continue
            candidates = sorted(n for n in names
                                if n.startswith(f'{wav_root}{sid}/') and n.endswith('_mic1.flac'))
            for member in candidates[:N_UTTS_PER_SPEAKER]:
                target = spk_dir / Path(member).name
                if target.exists():
                    continue
                zf.extract(member, extract_tmp)
                shutil.copy2(extract_tmp / member, target)
            if (i + 1) % 20 == 0 or (i + 1) == len(speakers_all):
                print('  extracted %d/%d speakers' % (i + 1, len(speakers_all)), flush=True)

    shutil.rmtree('/content/vctk_extract', ignore_errors=True)
    if VCTK_LOCAL_ZIP == Path('/content/VCTK-Corpus-0.92.zip') and VCTK_LOCAL_ZIP.exists():
        VCTK_LOCAL_ZIP.unlink()   # free local disk; the small selection above is what persists
    print('VCTK cache ready at %s' % VCTK_CACHE)

VCTK_SPEAKERS = parse_speaker_info(SPEAKER_INFO_CACHE)
usable_speakers = {sid: info for sid, info in VCTK_SPEAKERS.items()
                  if len(list((VCTK_CACHE / sid).glob('*_mic1.flac'))) >= N_UTTS_PER_SPEAKER}
excluded = len(VCTK_SPEAKERS) - len(usable_speakers)
gender_counts = {'M': sum(1 for v in usable_speakers.values() if v['gender'] == 'M'),
                 'F': sum(1 for v in usable_speakers.values() if v['gender'] == 'F')}

print('\nActual VCTK speaker count with >= %d cached mic1 utterances: %d (%d excluded for too few)'
      % (N_UTTS_PER_SPEAKER, len(usable_speakers), excluded))
print('Gender balance actually obtained: %s' % gender_counts)


## Step 5: WavLM-embed each VCTK speaker and apply the probe

For each speaker: read `N_UTTS_PER_SPEAKER` mic1 recordings, resample
48kHz -> 16kHz, RMS level-match each one exactly as the calibration corpus
was (`phase2b_subspace_embed.level_match`, same `TARGET_RMS`), extract WavLM
layers 3-5 mean+std, average the pooled features across utterances into one
6,144-dim vector, and apply `DEPLOYED_MODEL` to get that speaker's predicted
9-dim (or `RANK`-dim) coordinates in the preset-difference span. Checkpointed
to `VCTK_COEFFS_PATH` after every speaker -- a disconnect partway through
~110 speakers resumes rather than restarting, which is the actual reason
this notebook needs checkpointing (the calibration corpus above did not).

In [ ]:
# 10. Per-speaker WavLM features -> predicted preset-span coefficients.
import json as _json
import time

from scipy.signal import resample_poly

from phase2b_wavlm import rep_from

vctk_results = _json.loads(VCTK_COEFFS_PATH.read_text()) if VCTK_COEFFS_PATH.exists() else {}
todo = [sid for sid in sorted(usable_speakers) if sid not in vctk_results]
print('%d speakers already checkpointed, %d remaining' % (len(vctk_results), len(todo)))

t0 = time.time()
for i, sid in enumerate(todo):
    flac_paths = sorted((VCTK_CACHE / sid).glob('*_mic1.flac'))[:N_UTTS_PER_SPEAKER]
    per_utt_feats = []
    for p in flac_paths:
        wav48, sr = sf.read(str(p), dtype='float32')
        if wav48.ndim > 1:
            wav48 = wav48.mean(axis=1)
        assert sr == 48000, (p, sr)
        wav16 = resample_poly(wav48, 16000, 48000).astype(np.float32)
        leveled, rms_pre = level_match(wav16)
        if leveled is None:
            continue
        mean, std = pooled_features_gpu(wavlm_model, NORMALIZE, leveled, DEVICE)
        per_utt_feats.append(rep_from(mean[None], std[None], LAYERS, 'meanstd')[0])

    if not per_utt_feats:
        vctk_results[sid] = {'skipped': True, 'reason': 'every cached utterance was silent after level-matching'}
        continue

    feat = np.mean(np.stack(per_utt_feats, axis=0), axis=0, keepdims=True)   # (1, 6144)
    c_hat = DEPLOYED_MODEL.predict(DEPLOYED_SCALER.transform(feat))[0]
    vctk_results[sid] = {
        'gender': usable_speakers[sid]['gender'], 'age': usable_speakers[sid]['age'],
        'n_utterances_used': len(per_utt_feats), 'coefficients': c_hat.tolist(),
        'coefficient_norm': float(np.linalg.norm(c_hat)),
    }

    if (i + 1) % 5 == 0 or (i + 1) == len(todo):
        VCTK_COEFFS_PATH.write_text(_json.dumps(vctk_results, indent=2))
        elapsed = time.time() - t0
        print('[%d/%d] checkpointed (%.1fs elapsed, %.2fs/speaker)'
              % (i + 1, len(todo), elapsed, elapsed / (i + 1)), flush=True)

VCTK_COEFFS_PATH.write_text(_json.dumps(vctk_results, indent=2))
n_ok = sum(1 for v in vctk_results.values() if not v.get('skipped'))
print('\n%d/%d speakers have predicted coefficients (%d skipped as silent).'
      % (n_ok, len(vctk_results), len(vctk_results) - n_ok))


## Step 6: is the probe extrapolating?

The probe was trained on styles inside a small ball around one preset. This
cell reports where the ~110 predicted VCTK coefficients actually sit
relative to where the ten presets themselves sit, in the same 9-dim
coordinates. It does not merely compute a diagnostic and move on --if real
voices land far outside the presets' own range, that is printed as an
explicit warning, not left for the reader to notice in a table.

In [ ]:
# 11. Distribution check: VCTK predictions vs. the presets' own coefficients.
preset_matrix = np.stack([PRESET_COEFFS[p] for p in PRESETS])          # (10, RANK)
vctk_ids = sorted(sid for sid, v in vctk_results.items() if not v.get('skipped'))
vctk_matrix = np.stack([np.array(vctk_results[s]['coefficients']) for s in vctk_ids])  # (n, RANK)

print('Per-axis range, %d presets vs %d predicted VCTK speakers:' % (len(PRESETS), len(vctk_ids)))
for j in range(RANK):
    pc, vc = preset_matrix[:, j], vctk_matrix[:, j]
    print('  axis %d: presets [%+.3f, %+.3f]   VCTK predicted [%+.3f, %+.3f]  mean %+.3f std %.3f'
          % (j, pc.min(), pc.max(), vc.min(), vc.max(), vc.mean(), vc.std()))

preset_norms = np.linalg.norm(preset_matrix, axis=1)
vctk_norms = np.linalg.norm(vctk_matrix, axis=1)
print('\n||c|| presets:         %.3f - %.3f (mean %.3f)' % (preset_norms.min(), preset_norms.max(), preset_norms.mean()))
print('||c|| VCTK predicted:  %.3f - %.3f (mean %.3f)' % (vctk_norms.min(), vctk_norms.max(), vctk_norms.mean()))

frac_outside = float(np.mean(vctk_norms > preset_norms.max()))
print('\nFraction of VCTK speakers predicted OUTSIDE the presets\' own norm range: %.1f%%' % (100 * frac_outside))

if frac_outside > 0.2 or vctk_norms.mean() < 0.3 * preset_norms.mean():
    print('\nWARNING: predicted coefficients are either far outside the region the presets '
          'occupy, or compressed toward the training-ball center (ridge regression pulls '
          'uncertain predictions toward the training mean). Either pattern means the probe is '
          'very likely extrapolating past the eps=0.20 neighborhood it was fitted on. Treat the '
          'gate result in the next step as provisional pending further investigation -- e.g. a '
          'wider-eps calibration corpus, or checking prediction variance directly.')
else:
    print('\nPredicted VCTK coefficients broadly overlap the presets\' own range -- no immediate '
          'red flag, though this is a coarse per-axis/norm check, not proof that the probe '
          'transfers correctly to real recordings.')


## Step 7: the presentation-axis test, generalized to ~110 points

This is `py/phase3_presentation_stage1.py`'s own gate, **unmodified** --
`leave_one_out`, `class_mean_diff`, and `variance_explained` are imported
directly from that module. It hardcodes its ten presets as module-level
globals (`PRESETS`, `M_PRESETS`, `F_PRESETS`), read by those functions as
free variables, so re-running it against VCTK means repointing those three
globals at the VCTK speaker IDs and their gender labels rather than editing
or copying the functions. That is what the cell below does.

The functions expect each point as a flat `(24 active rows x 256)` vector in
raw `style_ttl` coordinates -- what the ten presets naturally are. A VCTK
speaker only has predicted coefficients in the 9-dim span, so those are
mapped back with the exact inverse of the mapping used everywhere else in
this project to turn coefficients into rows
(`phase2b_generate_subspace.sample_style`: `rows = unit_rows(P + c @ B)`),
applied here to the probe's *predicted* `c`, not a drawn sample.

In [ ]:
# 12. Reconstruct active-row vectors from predicted coefficients, then hand
# them to phase3_presentation_stage1's own functions, unchanged.
import phase3_presentation_stage1 as p3s1
from scipy.stats import binomtest


def reconstruct_active_ttl(c_hat, B, base_point):
    n_rows, n_cols = base_point.shape
    d = (c_hat @ B).reshape(n_rows, n_cols)
    rows = base_point + d
    rows = rows / np.linalg.norm(rows, axis=-1, keepdims=True).clip(min=1e-8)
    return rows.reshape(-1).astype(np.float64)


vecs_by_speaker = {sid: reconstruct_active_ttl(np.array(vctk_results[sid]['coefficients']), B_a, P)
                  for sid in vctk_ids}
m_ids = [s for s in vctk_ids if vctk_results[s]['gender'] == 'M']
f_ids = [s for s in vctk_ids if vctk_results[s]['gender'] == 'F']
print('%d usable speakers: %d M, %d F' % (len(vctk_ids), len(m_ids), len(f_ids)))

p3s1.PRESETS = vctk_ids
p3s1.M_PRESETS = m_ids
p3s1.F_PRESETS = f_ids

loo_records, n_correct = p3s1.leave_one_out(vecs_by_speaker)
n_total = len(vctk_ids)
accuracy = n_correct / n_total if n_total else float('nan')
pvalue = binomtest(n_correct, n_total, 0.5, alternative='greater').pvalue if n_total else float('nan')

print('\n=== Presentation-axis leave-one-out gate, VCTK (n=%d) ===' % n_total)
print('%d/%d correct (%.1f%%), one-sided binomial p=%.4g vs. chance'
      % (n_correct, n_total, 100 * accuracy, pvalue))
print('Ten-preset reference: 7/10 (70.0%%), p=0.172 (py/results/phase3/presentation_axis.json)')
gate_cleared = accuracy > 0.7
print('Gate (> 70%%, the preset-run\'s own threshold): %s' % ('CLEARED' if gate_cleared else 'NOT CLEARED'))

d_full, mean_f, mean_m, midpoint, f_train, m_train = p3s1.class_mean_diff(vecs_by_speaker, vctk_ids)
total_var, var_along_d, frac_var = p3s1.variance_explained(vecs_by_speaker, d_full)
print('\nVariance explained by the direction: %.1f%% (ten-preset reference: 20.3%%)' % (100 * frac_var))

gate_result = {
    'experiment': 'phase3_vctk_styles_colab presentation-axis gate',
    'n_total': n_total, 'n_m': len(m_ids), 'n_f': len(f_ids),
    'n_correct': n_correct, 'accuracy': accuracy, 'binomial_pvalue_vs_chance': pvalue,
    'gate_threshold': '> 70% (the ten-preset run\'s own 7/10 threshold)',
    'gate_cleared': gate_cleared,
    'variance_explained_fraction': frac_var,
    'ten_preset_reference': {'n_correct': 7, 'n_total': 10, 'pvalue': 0.172, 'variance_explained': 0.2028},
    'loo_records': loo_records,
    'distribution_check': {
        'fraction_vctk_outside_preset_norm_range': frac_outside,
        'preset_norm_mean': float(preset_norms.mean()), 'vctk_norm_mean': float(vctk_norms.mean()),
    },
}
GATE_RESULT_PATH.write_text(_json.dumps(gate_result, indent=2))
print('\nWrote %s' % GATE_RESULT_PATH)


## Path A: what to conclude

Read the gate result above next to the distribution-check warning from Step
6, not in isolation. If the gate clears **and** the distribution check found
no red flag, that is real evidence a presentation axis is locatable from
VCTK-scale data. If the gate clears but the distribution check warned about
extrapolation, the accuracy number is suspect -- a probe that is
compressing every prediction toward the training mean can still separate
two classes whose *true* means differ, by accident, without the individual
coefficients meaning anything. If the gate does not clear, that is itself
informative: either 110 points still are not enough, or the presentation
direction genuinely needs a wider or better-calibrated probe (the optional
Path B below) before it is locatable at all.

## Path B (optional): gradient refinement through the frozen graph

Everything below this point is gated behind `RUN_PATH_B` (set in the
Configuration cell, default `False`) and is not needed to answer the axis
question -- Path A already did that. Turn this on only if Path A's linear
readout is not accurate enough for a given use and per-speaker GPU time is
acceptable.

**What is verified going in, and what is not.** `py/onnx2torch_patches.py`
converts `text_encoder` to PyTorch and its self-test confirms the converted
graph agrees with `onnxruntime` (max abs diff < 1e-4) and that gradients
reach every element of `style_ttl`. That is the only one of the three graphs
this notebook needs (`text_encoder`, `vector_estimator`, `vocoder`) that has
ever been checked this way. The two patches themselves are generic
opset/spec-conformance fixes, not specific to `text_encoder`, so there is a
real chance they carry over cleanly -- but "a real chance" is not a
verification, which is why the next cell repeats the same numeric check
against all three converted graphs together, end to end, before any gradient
step is trusted.

**A duration-invariant loss, not a frame-aligned one.** The refinement text
is a fixed sentence, not the target speaker's actual words, so the
synthesized clip and the real recording never have matching duration or
content. Comparing them frame-by-frame would be meaningless. Instead the
loss compares per-mel-bin mean and standard deviation over time (a
duration-invariant "style" statistic, not a frame-wise spectrogram
distance) -- a deliberate simplification of "mel-spectrogram distance" as
named in this notebook's brief, and one more thing this path has not been
listened to yet.

**Timing is measured on whatever GPU this session actually has**, not
assumed from the 123-seconds-per-CPU-step figure cited above (different
hardware, different graph subset -- that figure covered only `text_encoder`
forward+backward, not a full denoising loop through three graphs).

In [ ]:
# 13. Path B setup: convert three graphs, cross-check the whole forward pass
# against onnxruntime end to end before trusting any gradient from it.
if not RUN_PATH_B:
    print('RUN_PATH_B is False -- skipping Path B entirely. Path A already answered the axis '
          'question (Step 7); this path only exists to refine individual speakers style_ttl '
          'past what the linear probe gives, if that is ever needed.')
else:
    import onnx
    from onnx2torch import convert as onnx2torch_convert
    from onnx2torch_patches import apply_patches

    GRAPH_NAMES = ['text_encoder', 'vector_estimator', 'vocoder']
    onnx_models, torch_graphs, torch_callers = {}, {}, {}

    def make_caller(onnx_model, torch_module):
        # onnx2torch is documented to generate forward(*inputs) in the graph's
        # declared input order. Reading that order back off the loaded
        # onnx.ModelProto, rather than hardcoding it per graph, means this does
        # not depend on guessing onnx2torch's positional convention correctly.
        input_names = [i.name for i in onnx_model.graph.input]

        def call(**kwargs):
            missing = [n for n in input_names if n not in kwargs]
            if missing:
                raise KeyError('missing inputs for this graph: %s (declared order: %s)'
                               % (missing, input_names))
            out = torch_module(*[kwargs[n] for n in input_names])
            return out[0] if isinstance(out, (tuple, list)) else out
        return call

    for name in GRAPH_NAMES:
        model = onnx.load(str(Path(ONNX_DIR) / (name + '.onnx')))
        summary = apply_patches(model)
        print('%s: %s' % (name, summary))
        if summary.unresolvable:
            raise RuntimeError('%s has unresolvable ops: %s -- onnx2torch_patches does not '
                               'cover this graph as written; Path B cannot proceed without '
                               'extending that module.' % (name, summary.unresolvable))
        tm = onnx2torch_convert(model)
        tm.eval().to(DEVICE)
        onnx_models[name], torch_graphs[name] = model, tm
        torch_callers[name] = make_caller(model, tm)

    print('\nConverted: %s' % list(torch_graphs))
    print('Only text_encoder has prior numerical verification in this repo -- the next cell '
          'checks all three together before any gradient step runs.')


In [ ]:
# 14. End-to-end numerical cross-check: torch (all three converted graphs)
# vs. onnxruntime, same style, same text, same seeded latent. This is the
# first time vector_estimator and vocoder are checked this way anywhere in
# this repo -- do not proceed to gradients if it fails.
if not RUN_PATH_B:
    print('RUN_PATH_B is False -- skipping.')
else:
    from helper import get_latent_mask

    CHECK_TEXT = 'The quick brown fox jumps over the lazy dog.'
    CHECK_SEED = 0

    def onnx_infer_reference(style):
        wav, dur = tts(CHECK_TEXT, 'en', style, TOTAL_STEP_PATHB, 1.0, seed=CHECK_SEED)
        return wav, dur

    def torch_infer_wav(style_ttl_t, dp_np, text, total_step, speed, seed, tts_ref, device):
        text_ids, text_mask = tts_ref.text_processor([text], ['en'])
        dur_onnx, *_ = tts_ref.dp_ort.run(
            None, {'text_ids': text_ids, 'style_dp': dp_np, 'text_mask': text_mask})
        dur_onnx = dur_onnx / speed
        text_ids_t = torch.tensor(text_ids, device=device)
        text_mask_t = torch.tensor(text_mask, device=device)
        text_emb_t = torch_callers['text_encoder'](
            text_ids=text_ids_t, style_ttl=style_ttl_t, text_mask=text_mask_t)

        bsz = 1
        wav_len_max = dur_onnx.max() * tts_ref.sample_rate
        wav_lengths = (dur_onnx * tts_ref.sample_rate).astype(np.int64)
        chunk_size = tts_ref.base_chunk_size * tts_ref.chunk_compress_factor
        latent_len = int(np.ceil(wav_len_max / chunk_size))
        latent_dim = tts_ref.ldim * tts_ref.chunk_compress_factor
        rng = np.random.default_rng(seed)
        noisy_latent_np = rng.standard_normal((bsz, latent_dim, latent_len)).astype(np.float32)
        latent_mask_np = get_latent_mask(wav_lengths, tts_ref.base_chunk_size, tts_ref.chunk_compress_factor)
        noisy_latent_np = noisy_latent_np * latent_mask_np
        xt = torch.tensor(noisy_latent_np, device=device)
        latent_mask_t = torch.tensor(latent_mask_np, device=device)

        for step in range(total_step):
            current_step_t = torch.tensor([step] * bsz, dtype=torch.float32, device=device)
            total_step_t = torch.tensor([total_step] * bsz, dtype=torch.float32, device=device)
            xt = torch_callers['vector_estimator'](
                noisy_latent=xt, text_emb=text_emb_t, style_ttl=style_ttl_t,
                text_mask=text_mask_t, latent_mask=latent_mask_t,
                current_step=current_step_t, total_step=total_step_t)
        wav_t = torch_callers['vocoder'](latent=xt)
        return wav_t, dur_onnx

    TOTAL_STEP_PATHB = 8
    check_style = all_presets[BASE_PRESET]
    wav_ref, dur_ref = onnx_infer_reference(check_style)

    style_ttl_check = torch.tensor(check_style.ttl, device=DEVICE)
    with torch.no_grad():
        wav_torch, dur_torch = torch_infer_wav(
            style_ttl_check, check_style.dp, CHECK_TEXT, TOTAL_STEP_PATHB, 1.0, CHECK_SEED, tts, DEVICE)
    wav_torch_np = wav_torch.detach().cpu().numpy()

    n = min(wav_ref.shape[-1], wav_torch_np.shape[-1])
    max_diff = float(np.abs(wav_ref[..., :n] - wav_torch_np[..., :n]).max())
    print('onnxruntime vs. torch (3 converted graphs), same seed: max abs waveform diff = %.3e' % max_diff)
    PATH_B_CROSS_CHECK_OK = max_diff < 1e-3
    if not PATH_B_CROSS_CHECK_OK:
        print('WARNING: the converted graphs disagree with onnxruntime by more than the loose '
              '1e-3 tolerance used here (text_encoder alone verified to 1e-4; three graphs in a '
              'chain accumulate more numerical drift, so this bound is deliberately looser). Do '
              'not trust gradients from this conversion until the disagreement is understood -- '
              'inspect vector_estimator and vocoder individually the way onnx2torch_patches.py '
              'own self-test does for text_encoder.')
    else:
        print('Cross-check passed within tolerance.')

    # Gradient reaches style_ttl end to end?
    style_ttl_grad_check = torch.tensor(check_style.ttl, device=DEVICE, requires_grad=True)
    wav_g, _ = torch_infer_wav(style_ttl_grad_check, check_style.dp, CHECK_TEXT, TOTAL_STEP_PATHB,
                              1.0, CHECK_SEED, tts, DEVICE)
    wav_g.sum().backward()
    grad = style_ttl_grad_check.grad
    grad_ok = grad is not None and torch.isfinite(grad).all().item() and (grad != 0).any().item()
    print('Gradient reaches style_ttl end to end through all three graphs: %s (nonzero=%d/%d)'
          % (grad_ok, int((grad != 0).sum().item()) if grad is not None else 0,
             grad.numel() if grad is not None else 0))
    PATH_B_CROSS_CHECK_OK = PATH_B_CROSS_CHECK_OK and grad_ok


In [ ]:
# 15. Mel-statistic loss and the refinement step. Active rows only are
# trainable (the 26 inactive rows stay frozen, matching every other
# active-row-restricted fit in this project); every row is re-projected to
# unit norm after each step, matching the optimizer kdrkdrkdr/supertonic.embed
# itself uses (new-plan.md, Phase 1a).
if not RUN_PATH_B:
    print('RUN_PATH_B is False -- skipping.')
elif not PATH_B_CROSS_CHECK_OK:
    print('Cross-check in the previous cell did not pass -- refusing to run gradient steps on '
          'an unverified conversion. Fix that before enabling this cell.')
else:
    import torchaudio

    REFINE_TEXT = 'The quick brown fox jumps over the lazy dog.'
    REFINE_SPEED = 1.0    # this fork default -- no compatibility constraint here, unlike the
                          # calibration corpus above which intentionally pins 1.05
    REFINE_TOTAL_STEP = 8
    REFINE_SEED = 0
    REFINE_LR = 0.02

    MEL_TRANSFORM = torchaudio.transforms.MelSpectrogram(
        sample_rate=tts.sample_rate, n_fft=1024, hop_length=256, n_mels=80).to(DEVICE)

    def mel_stats(wav_t):
        mel = MEL_TRANSFORM(wav_t)
        log_mel = torch.log(mel.clamp(min=1e-5))
        return log_mel.mean(dim=-1), log_mel.std(dim=-1)

    def mel_style_loss(wav_pred_t, wav_target_t):
        mp, sp = mel_stats(wav_pred_t)
        mt, st = mel_stats(wav_target_t)
        return torch.mean((mp - mt) ** 2) + torch.mean((sp - st) ** 2)

    def refine_style(c0, target_wav_16k, n_steps, lr=REFINE_LR):
        rows0 = (c0 @ B_a).reshape(P.shape) + P
        rows0 = rows0 / np.linalg.norm(rows0, axis=-1, keepdims=True).clip(min=1e-8)

        base_full = torch.tensor(base_ttl, device=DEVICE)               # (1, 50, 256), frozen
        active_param = torch.tensor(rows0[None].astype(np.float32), device=DEVICE, requires_grad=True)
        target_t = torch.tensor(target_wav_16k[None].astype(np.float32), device=DEVICE)

        opt = torch.optim.Adam([active_param], lr=lr)
        losses = []
        for step in range(n_steps):
            opt.zero_grad()
            full = base_full.clone()
            full[:, ACTIVE_ROWS, :] = active_param
            wav_t, _ = torch_infer_wav(full, dp_ref, REFINE_TEXT, REFINE_TOTAL_STEP, REFINE_SPEED,
                                       REFINE_SEED, tts, DEVICE)
            loss = mel_style_loss(wav_t, target_t)
            loss.backward()
            opt.step()
            with torch.no_grad():
                active_param.data = active_param.data / active_param.data.norm(dim=-1, keepdim=True).clamp(min=1e-8)
            losses.append(float(loss.item()))
        with torch.no_grad():
            full = base_full.clone()
            full[:, ACTIVE_ROWS, :] = active_param
        return full.detach().cpu().numpy(), losses

    print('refine_style defined (%d active rows trainable, %d frozen).' % (len(ACTIVE_ROWS), 50 - len(ACTIVE_ROWS)))


### Timing probe: measured on this session's GPU, not assumed

One speaker, a handful of steps, wall-clock timed here -- not the
123-seconds-per-CPU-step figure from the opening markdown, which covered a
different graph subset (`text_encoder` alone) on different hardware
entirely. Extrapolated from this measurement, not from that one.

In [ ]:
# 16. Time a handful of gradient steps on one speaker, extrapolate.
if not RUN_PATH_B:
    print('RUN_PATH_B is False -- skipping.')
elif not PATH_B_CROSS_CHECK_OK:
    print('Cross-check did not pass -- skipping the timing probe too.')
else:
    import time as _time

    timing_sid = vctk_ids[0]
    flac0 = sorted((VCTK_CACHE / timing_sid).glob('*_mic1.flac'))[0]
    wav48, sr = sf.read(str(flac0), dtype='float32')
    if wav48.ndim > 1:
        wav48 = wav48.mean(axis=1)
    wav16 = resample_poly(wav48, 16000, 48000).astype(np.float32)
    leveled, _ = level_match(wav16)
    c0 = np.array(vctk_results[timing_sid]['coefficients'])

    N_TIMING_STEPS = 3
    t0 = _time.time()
    _, timing_losses = refine_style(c0, leveled if leveled is not None else wav16, N_TIMING_STEPS)
    elapsed = _time.time() - t0
    s_per_step = elapsed / N_TIMING_STEPS

    print('Measured on this GPU (%s): %.1fs for %d steps -> %.1f s/step'
          % (torch.cuda.get_device_name(0), elapsed, N_TIMING_STEPS, s_per_step))
    print('Loss trajectory: %s' % [round(l, 4) for l in timing_losses])
    print('\nExtrapolated (linear in step count -- not independently verified past 3 steps):')
    print('  one speaker, %d steps:  ~%.1f min' % (PATH_B_N_STEPS, s_per_step * PATH_B_N_STEPS / 60))
    print('  %d speakers, %d steps each:  ~%.1f h'
          % (PATH_B_MAX_SPEAKERS, PATH_B_N_STEPS, s_per_step * PATH_B_N_STEPS * PATH_B_MAX_SPEAKERS / 3600))
    print('  all ~110 speakers, %d steps each:  ~%.1f h (extrapolation, not a plan -- '
          'PATH_B_MAX_SPEAKERS bounds what the next cell actually runs)'
          % (PATH_B_N_STEPS, s_per_step * PATH_B_N_STEPS * len(vctk_ids) / 3600))


### The refinement loop itself, bounded and resumable

Runs at most `PATH_B_MAX_SPEAKERS` speakers (default 3) for `PATH_B_N_STEPS`
steps each (default 30) -- turning `RUN_PATH_B` on never silently commits to
all ~110 speakers. Checkpointed per speaker to Drive, same pattern as Path
A, since even this bounded run can span more than one session at the extrapolated
per-step cost above.

In [ ]:
# 17. Bounded, resumable Path B refinement loop.
if not RUN_PATH_B:
    print('RUN_PATH_B is False -- skipping. Path A (Step 7) already answered the axis question; '
          'this loop only exists to refine individual speakers past the linear probe, bounded '
          'to PATH_B_MAX_SPEAKERS=%d speakers so enabling it never commits to all ~110.'
          % PATH_B_MAX_SPEAKERS)
elif not PATH_B_CROSS_CHECK_OK:
    print('Cross-check did not pass -- refusing to run the full loop.')
else:
    pathb_results_path = PATHB_DIR / 'refined_styles.json'
    pathb_results = _json.loads(pathb_results_path.read_text()) if pathb_results_path.exists() else {}

    targets = [s for s in vctk_ids if not vctk_results[s].get('skipped')][:PATH_B_MAX_SPEAKERS]
    todo_b = [s for s in targets if s not in pathb_results]
    print('%d/%d target speakers already refined; %d remaining' % (len(targets) - len(todo_b), len(targets), len(todo_b)))

    for i, sid in enumerate(todo_b):
        flac0 = sorted((VCTK_CACHE / sid).glob('*_mic1.flac'))[0]
        wav48, sr = sf.read(str(flac0), dtype='float32')
        if wav48.ndim > 1:
            wav48 = wav48.mean(axis=1)
        wav16 = resample_poly(wav48, 16000, 48000).astype(np.float32)
        leveled, _ = level_match(wav16)
        target_wav = leveled if leveled is not None else wav16
        c0 = np.array(vctk_results[sid]['coefficients'])

        t0 = time.time()
        refined_ttl, losses = refine_style(c0, target_wav, PATH_B_N_STEPS)
        elapsed = time.time() - t0

        out_path = PATHB_DIR / (sid + '_style_ttl.npy')
        np.save(out_path, refined_ttl)
        pathb_results[sid] = {
            'style_ttl_path': str(out_path), 'n_steps': PATH_B_N_STEPS,
            'loss_first': losses[0], 'loss_last': losses[-1], 'elapsed_sec': elapsed,
            'warm_start_coefficients': c0.tolist(),
        }
        pathb_results_path.write_text(_json.dumps(pathb_results, indent=2))
        print('[%d/%d] %s: loss %.4f -> %.4f in %.1fs, checkpointed'
              % (i + 1, len(todo_b), sid, losses[0], losses[-1], elapsed), flush=True)

    print('\nWrote %s' % pathb_results_path)


## What this notebook does and does not settle

**It settles** whether a presentation axis is locatable once the sample size
moves from ten labelled presets to roughly a hundred labelled VCTK speakers,
using the exact leave-one-out gate the ten-preset run used, on coefficients
read off real recordings by a probe whose validated accuracy (0.9376 mean
R^2 against a 0.7593 random-direction control) is established elsewhere in
this repo. Step 6's distribution check is the built-in honesty check on
whether that probe's accuracy actually transfers to voices it never saw
synthesized, rather than an assumption this notebook asks the reader to
grant.

**It does not settle** the age or emotion axes (VCTK carries no usable
signal for either -- see `new-plan.md` Phase 4), whether the presentation
direction found here is genuinely orthogonal to identity rather than
partially confounded with it (no orthogonalization is attempted in this
notebook -- that is `new-plan.md` Phase 3's Gram-Schmidt step, downstream of
this result), or whether Path B's gradient refinement actually improves
per-speaker fidelity (its numerical cross-check may not even pass on the
GPU this session gets, and its mel-statistic loss is a stated
simplification of true perceptual distance).

**What could not be verified from the repo alone, stated plainly:** the VCTK
fetch cell has not been executed against a live Colab runtime as part of
authoring this notebook, so the exact DataShare URL, its internal zip
layout, and the true per-speaker mic1 availability are all best-effort
rather than confirmed; the manual-upload fallback exists for exactly that
reason. Every timing number Path B reports is measured inside its own cell
on whatever GPU that session actually has -- none are assumed from the
123-second CPU figure this notebook opens with, which covers a different
graph subset on different hardware.

In [ ]:
# 18. Bundle results for download. Audio (calibration corpus, VCTK cache) and
# the deployed probe's raw feature arrays stay on Drive -- gitignored in this
# repo and unneeded by whoever reads the JSON. The JSON is the deliverable:
# per-speaker coefficients and labels the user can hand back.
import shutil

from google.colab import files

bundle = Path('/content/phase3-vctk-results')
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir()

shutil.copy2(VCTK_COEFFS_PATH, bundle / 'vctk_coefficients.json')
shutil.copy2(GATE_RESULT_PATH, bundle / 'presentation_axis_vctk.json')
if (PATHB_DIR / 'refined_styles.json').exists():
    shutil.copy2(PATHB_DIR / 'refined_styles.json', bundle / 'pathb_refined_styles.json')

manifest_summary = {
    'basis_rank': RANK, 'base_preset': BASE_PRESET, 'active_rows': ACTIVE_ROWS,
    'n_per_cond_calibration': N_PER_COND, 'layers_1based': LAYERS,
    'n_utts_per_speaker': N_UTTS_PER_SPEAKER,
    'n_vctk_speakers_usable': len(vctk_ids), 'n_vctk_speakers_total_labelled': len(VCTK_SPEAKERS),
    'gender_balance': gender_counts,
    'calibration_probe_r2_this_run': float(np.nanmean(comp_r2)),
    'calibration_probe_r2_reference': REFERENCE_A['per_component_r2_mean'],
}
_json.dump(manifest_summary, open(bundle / 'run_summary.json', 'w'), indent=2)

archive = shutil.make_archive('/content/phase3-vctk-results', 'zip', str(bundle))
print('Archive: %s (%.1f KB)' % (archive, Path(archive).stat().st_size / 1e3))
print('Drive copy of everything (calibration audio, VCTK cache, checkpoints): %s' % WORKSPACE)
files.download(archive)
